In [ ]:
import datetime
import os
import ee
import geemap
import joblib
import matplotlib.pyplot as plt

from identify_locations import identify_forests, identify_route_buffer
from read_and_process_hls import compute_hls_indices
from fit_greendown_curves import compute_transition_dates, compute_average_transition_dates
from filter_ci_widths import count_narrow_ci_pixel_years
from build_data_table import build_feature_table, export_prediction_avg_assets
from edit_data_table import edit_feature_table
from plot_feature_distributions import plot_feature_distributions
from decision_trees import split_data, fit_tree
from gridmet_utils import fetch_gridmet_cdd_historical
from constants import DATA_DIR, GREENDOWN_DIR, MODEL_DIR

from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error
from sklearn import tree

ee.Initialize(project='turnkey-lacing-391919')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(GREENDOWN_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

ma_forest    = identify_forests()
route_buffer = identify_route_buffer()

start_year    = 2013 #2013  # First HLS year
download_year = 2025  # Last year to download (includes 2025 for prediction)
train_year    = 2024  # Last year included in model training

# ----------------------------
# Fit logistic curves for each year (download all years including 2025)
# ----------------------------
all_year_paths = []
for y in range(start_year, download_year + 1):
    print(f'Processing {y}...')
    hls   = compute_hls_indices(route_buffer, ma_forest, y)
    paths = compute_transition_dates(hls, route_buffer, ma_forest, y, data_dir=DATA_DIR, greendown_dir=GREENDOWN_DIR)
    all_year_paths.append(paths)

prev_year_paths = all_year_paths[-1]   # most recent year

# ----------------------------
# Average transition dates across all years
# ----------------------------
print('Computing averages...')
avg_paths = compute_average_transition_dates(all_year_paths, data_dir=DATA_DIR, greendown_dir=GREENDOWN_DIR)

# ----------------------------
# Export committed assets the live (Action) prediction needs for
# doy_minus_avg_middle: the CI-filtered per-pixel cross-year average and the
# global gap-fill scalar. Re-run on every retrain, then commit:
#   greendown_middle_avg_filtered.tif, greendown_avg_meta.json
# ----------------------------
print('\nExporting prediction average assets...')
export_prediction_avg_assets(DATA_DIR, GREENDOWN_DIR)

# ----------------------------
# Filter: pixels with CI width < 15 days for all transitions
# ----------------------------
print('\nFiltering pixel-years by CI width...')
years = list(range(start_year, download_year + 1))
count_narrow_ci_pixel_years(GREENDOWN_DIR, years)


Prepare Data

In [ ]:
# ----------------------------
# Download gridMET cold degree-days for training years (stored locally).
# Skips any year whose gridmet_cdd_{year}.npz is already cached.
# Must be run before build_feature_table so the cdd_accumulated feature
# can be populated. Re-running is safe — cached years are skipped.
# ----------------------------
print('\nDownloading gridMET CDD for training years...')
training_years = list(range(start_year, train_year + 1))
for y in training_years:
    fetch_gridmet_cdd_historical(y, route_buffer, DATA_DIR)

# ----------------------------
# Build labeled EVI/NDVI feature table (training years only: 2013–2024)
# ----------------------------
print('\nBuilding labeled feature table...')
feature_df = build_feature_table(DATA_DIR, GREENDOWN_DIR, training_years)
#plot_feature_distributions(feature_df)
feature_df_edited = edit_feature_table(feature_df, GREENDOWN_DIR, MODEL_DIR)
print(feature_df_edited.head())
print('\nNaN counts per column:')
print(feature_df_edited.isna().sum())
#plot_feature_distributions(feature_df_edited)


In [ ]:
# ----------------------------
# Build decision tree
# ----------------------------
x_train, x_test, y_train, y_test = split_data(feature_df_edited)

mdl = fit_tree(x_train, y_train, True)

# Save model for dashboard use
joblib.dump(mdl, os.path.join(MODEL_DIR, 'decision_tree_model.joblib'))
print(f'Model saved to {MODEL_DIR}/decision_tree_model.joblib')

# ----------------------------
# Visualize decision tree
# ----------------------------
fig, ax = plt.subplots(figsize=(40, 20))
tree.plot_tree(mdl, feature_names=list(x_train.columns), max_depth = 3, filled=True, ax=ax)
fig.savefig(os.path.join(MODEL_DIR, 'decision_tree.png'), dpi=150, bbox_inches='tight')
plt.show()

# Plot feature importances
importances = mdl.feature_importances_
feature_names = list(x_train.columns)
sorted_idx = sorted(range(len(importances)), key=lambda i: importances[i], reverse=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(feature_names) * 0.4)))
ax.barh([feature_names[i] for i in sorted_idx], [importances[i] for i in sorted_idx])
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Decision Tree Feature Importances')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(os.path.join(MODEL_DIR, 'feature_importances.png'), dpi=150, bbox_inches='tight')
plt.show()

# ----------------------------
# Test on test data
# ----------------------------
y_pred = mdl.predict(x_test)

# ----------------------------
# Evaluate model
# ----------------------------
print('\nTest Accuracy Score:')
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

#Check for overfitting by comparing accuracy of predicting training data to predicting test data
y_pred_train = mdl.predict(x_train)
print('\nTrain Accuracy Score:')
print(accuracy_score(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))
#Take-away: without pruning the accuracy on the training data is 1 and the accuracy on the testing data in 0.75 so the model is overfitting
    #With pruning the test accuracy is 0.74 and training is 0.78 so much closer and less overfitting


In [ ]:
# ----------------------------
# RNN Training
# Builds a separate feature table with pixel_id and temperature features
# retained, reshapes into per-pixel-year sequences, trains an LSTM model,
# and saves rnn_model.pt + rnn_norm_stats.json + rnn_model_config.json.
# The decision tree (Cell 1) is unchanged; the website continues to use it.
# ----------------------------
import torch
import numpy as np
from rnn_model import (
    RNN_FEATURE_COLS, RNNPhenologyModel,
    build_rnn_sequences, compute_rnn_norm_stats, normalize_sequences,
    split_sequences, oversample_sequences, train_rnn, evaluate_rnn, save_rnn_model,
)

# Rebuild feature table with pixel_id retained and temperature features enabled.
# Reuses already-downloaded HLS stacks and gridMET cache — no new GEE calls.
print('Building RNN feature table (with pixel_id + temperature)...')
rnn_df = build_feature_table(DATA_DIR, GREENDOWN_DIR, training_years,
                              retain_pixel_id=True,
                              include_temperature=True)
n_pixels = rnn_df['pixel_id'].nunique()
print(f'  {len(rnn_df)} observations across {n_pixels} unique pixels')
print(f'  NaN counts:\n{rnn_df[RNN_FEATURE_COLS].isna().sum().to_string()}')

# Reshape into per-pixel-year sequences (one sequence per pixel per year)
sequences = build_rnn_sequences(rnn_df)
seq_lens  = [len(s[0]) for s in sequences]
print(f'\n  {len(sequences)} sequences  |  '
      f'length range: {min(seq_lens)}–{max(seq_lens)}  |  '
      f'median: {int(sorted(seq_lens)[len(seq_lens)//2])}')

# Compute normalization stats and normalize in-place.
# normalize_sequences replaces any remaining NaN with 0.0 (= feature mean
# in z-score space), so NaN in temperature features cannot reach the LSTM.
norm_stats = compute_rnn_norm_stats(sequences)
normalize_sequences(sequences, norm_stats)
nan_check = sum(np.isnan(f).any() for f, _ in sequences)
print(f'  Sequences with NaN after normalization: {nan_check}  (must be 0)')

# Three-way split: 70% train / 15% val / 15% test.
# Test set is held out entirely until final evaluation — val drives early stopping.
# split_sequences splits by pixel-year so no pixel leaks across splits.
trainval_seqs, test_seqs  = split_sequences(sequences,     val_frac=0.15, seed=42)
train_seqs,    val_seqs   = split_sequences(trainval_seqs, val_frac=0.176, seed=42)
# 0.176 × 85% ≈ 15% of total
print(f'  Train: {len(train_seqs)}  |  Val: {len(val_seqs)}  |  Test: {len(test_seqs)} sequences')

# Oversample minority-class sequences in the training set only.
# target_ratio=1.0 fully balances all classes to the majority count.
# Lower (e.g. 0.5) if you see overfitting on the val classification report.
train_seqs = oversample_sequences(train_seqs, target_ratio=1.0)
print(f'  After oversampling: {len(train_seqs)} training sequences')

# Build and train model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nTraining on {device}...')
model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
model = train_rnn(model, train_seqs, val_seqs,
                  epochs=60, lr=1e-3, batch_size=64, device=device)

# ----------------------------
# Evaluate: val, test, and train (train last to check for overfitting)
# ----------------------------
print('\nValidation results:')
evaluate_rnn(model, val_seqs, device=device)

print('\nTest results:')
evaluate_rnn(model, test_seqs, device=device)

print('\nTrain results (compare to test to check overfitting):')
evaluate_rnn(model, train_seqs, device=device)

# Save
save_rnn_model(model, norm_stats, MODEL_DIR)
